# 작업 5.2: Qiskit Primitives V1에서 V2로 마이그레이션하는 가이드

이 노트북은 Qiskit의 V1 Primitives에서 최신 V2 Primitives(`EstimatorV2`, `SamplerV2`)로 코드를 업데이트하는 실습형 가이드를 제공합니다.

**참고:** 최근 Qiskit 버전에서 V1 Primitives는 더 이상 권장되지 않습니다. 이 가이드에서는 V1 코드 패턴을 **주석 처리된 예시**로 보여주고, 로컬 `AerSimulator`에서 실행 가능한 **동작하는 V2 코드**를 함께 제공합니다.

## 설정: import와 백엔드

먼저 V2 primitives에 필요한 모든 것을 가져오겠습니다. 모든 예시는 로컬 `AerSimulator`에서 실행됩니다.

In [1]:
# 일반적인 Qiskit 도구
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
import numpy as np

# 모든 예시에서 사용할 로컬 시뮬레이터 백엔드
from qiskit_aer import AerSimulator

# --- V2 Primitives (현재/권장 방식) ---
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import SamplerV2 as Sampler

# 옵션 딕셔너리를 보기 좋게 출력하는 도구
from dataclasses import asdict

# 전체 예시에서 사용할 로컬 백엔드 인스턴스 정의
local_backend = AerSimulator()

print("import가 완료되었고 로컬 백엔드가 준비되었습니다.")

import가 완료되었고 로컬 백엔드가 준비되었습니다.


In [2]:
# Bell 상태 회로를 만드는 보조 함수
def create_bell_circuit():
    """Bell 상태(최대 얽힘 상태) 회로를 생성합니다."""
    qc = QuantumCircuit(2)
    qc.h(0)  # 큐비트 0에 Hadamard 적용
    qc.cx(0, 1)  # control=0, target=1인 CNOT 적용
    return qc

# 테스트
bell = create_bell_circuit()
print(bell)

     ┌───┐     
q_0: ┤ H ├──■──
     └───┘┌─┴─┐
q_1: ─────┤ X ├
          └───┘


## 1. `run()` 메서드: 회로, 관측량, 파라미터

V1에서 V2로 바뀌면서 가장 중요한 변화는 `run()` 메서드 입력 구조입니다.

### 핵심 차이점:
- **V1:** `circuits`, `observables`, `parameter_values`를 각각 별도 인자로 받음
- **V2:** 회로와 관측량, 파라미터를 묶은 튜플들의 단일 리스트인 **PUBs (Primitive Unified Blocs)** 를 받음

### Estimator: V1 → V2 마이그레이션

간단한 Bell 회로 기대값 계산을 어떻게 옮기는지 살펴보겠습니다.

In [3]:
print("=" * 60)
print("V1 코드 (더 이상 권장되지 않음 - 참고용):")
print("=" * 60)
print("""
# V1 Estimator - 예전 방식 (이제는 사용하지 마세요)
from qiskit_ibm_runtime import Estimator

circuit = create_bell_circuit()
observable = SparsePauliOp("ZZ")

estimator = Estimator(backend=local_backend)
# V1은 키워드 인자를 각각 따로 사용했습니다
job = estimator.run(
    circuits=[circuit],
    observables=[observable]
)
result = job.result()
print(f"기대값: {result.values[0]}")
""")

print("\n" + "=" * 60)
print("V2 코드 (현재 방식 - 실행 가능 예시):")
print("=" * 60)

V1 코드 (더 이상 권장되지 않음 - 참고용):

# V1 Estimator - 예전 방식 (이제는 사용하지 마세요)
from qiskit_ibm_runtime import Estimator

circuit = create_bell_circuit()
observable = SparsePauliOp("ZZ")

estimator = Estimator(backend=local_backend)
# V1은 키워드 인자를 각각 따로 사용했습니다
job = estimator.run(
    circuits=[circuit],
    observables=[observable]
)
result = job.result()
print(f"기대값: {result.values[0]}")


V2 코드 (현재 방식 - 실행 가능 예시):


In [4]:
# V2 Estimator - 새 방식 (이 방식을 사용하세요!)
from qiskit_ibm_runtime import EstimatorV2 as Estimator

circuit = create_bell_circuit()
observable = SparsePauliOp("ZZ")

estimator = Estimator(mode=local_backend)

# V2는 PUBs를 사용합니다: (circuit, observable, [선택적 params]) 튜플들의 리스트
job = estimator.run(pubs=[(circuit, observable)])
result = job.result()

# V2 결과는 PUB 인덱스로 접근하며, 데이터는 .data 속성에 있습니다
pub_result = result[0]
print(f"V2 작업 ID: {job.job_id()}")
print(f"V2 기대값: {pub_result.data.evs}")
print(f"V2 표준편차: {pub_result.data.stds}")

V2 작업 ID: d1eaf0ea-8ce2-4add-90d3-4df0c3ffedb2
V2 기대값: 1.0
V2 표준편차: 0.0


### Sampler: V1 → V2 마이그레이션

Sampler도 같은 패턴을 따릅니다.

In [5]:
print("=" * 60)
print("V1 코드 (더 이상 권장되지 않음 - 참고용):")
print("=" * 60)
print("""
# V1 Sampler - 예전 방식
from qiskit_ibm_runtime import Sampler

circuit = create_bell_circuit()
circuit.measure_all()

sampler = Sampler(backend=local_backend)
# V1은 키워드 인자를 각각 따로 사용했습니다
job = sampler.run(circuits=[circuit])
result = job.result()
print(f"준확률 분포: {result.quasi_dists[0]}")
""")

print("\n" + "=" * 60)
print("V2 코드 (현재 방식 - 실행 가능 예시):")
print("=" * 60)

V1 코드 (더 이상 권장되지 않음 - 참고용):

# V1 Sampler - 예전 방식
from qiskit_ibm_runtime import Sampler

circuit = create_bell_circuit()
circuit.measure_all()

sampler = Sampler(backend=local_backend)
# V1은 키워드 인자를 각각 따로 사용했습니다
job = sampler.run(circuits=[circuit])
result = job.result()
print(f"준확률 분포: {result.quasi_dists[0]}")


V2 코드 (현재 방식 - 실행 가능 예시):


In [6]:
# V2 Sampler - 새 방식 (이 방식을 사용하세요!)
from qiskit_ibm_runtime import SamplerV2 as Sampler

circuit = create_bell_circuit()
circuit.measure_all()

sampler = Sampler(mode=local_backend)

# V2는 PUBs를 사용합니다: (circuit, [선택적 params], [선택적 shots]) 튜플들의 리스트
# 파라미터가 없는 단일 회로 - 튜플로 만들기 위해 쉼표에 주의하세요
job = sampler.run(pubs=[(circuit,)])
result = job.result()

# V2 결과는 PUB 인덱스로 접근합니다
pub_result = result[0]
print(f"V2 작업 ID: {job.job_id()}")
print(f"V2 비트스트링 카운트: {pub_result.data.meas.get_counts()}")
print(f"처음 5개 비트스트링: {pub_result.data.meas.get_bitstrings()[:5]}")

V2 작업 ID: 7ad298b5-3b7d-4942-94f3-568b1b4a350c
V2 비트스트링 카운트: {'11': 496, '00': 528}
처음 5개 비트스트링: ['11', '11', '11', '11', '00']


## 2. 서로 다른 파라미터를 갖는 여러 작업 다루기

V2의 PUB 시스템은 다양한 작업을 훨씬 쉽게 실행하게 해줍니다. 각 회로마다 서로 다른 shots, 파라미터, 관측량을 지정할 수 있습니다.

### 예시: 서로 다른 shot 수를 가진 두 개의 회로

In [7]:
# 서로 다른 두 회로 생성
circuit1 = create_bell_circuit()
circuit1.measure_all()

circuit2 = create_bell_circuit()
circuit2.ry(np.pi / 4, 0)  # 추가 회전 적용
circuit2.measure_all()

# Sampler용 V2 PUB 형식: (circuit, parameter_values, shots)
# 각 회로마다 서로 다른 shot 수를 지정할 수 있습니다!
pub1 = (circuit1, None, 100)   # 회로 1: 100 shots
pub2 = (circuit2, None, 500)   # 회로 2: 500 shots

sampler = Sampler(mode=local_backend)
job = sampler.run(pubs=[pub1, pub2])
result = job.result()

# 결과는 입력 PUB 순서에 대응하는 인덱스로 접근합니다
print(f"회로 1 결과 (shots={result[0].metadata['shots']}):\n  {result[0].data.meas.get_counts()}")
print(f"\n회로 2 결과 (shots={result[1].metadata['shots']}):\n  {result[1].data.meas.get_counts()}")

회로 1 결과 (shots=100):
  {'11': 45, '00': 55}

회로 2 결과 (shots=500):
  {'01': 36, '11': 191, '00': 239, '10': 34}


### 예시: 하나의 회로에서 여러 관측량 측정하기

V2에서는 같은 회로에 대해 여러 관측량을 쉽게 측정할 수 있습니다.

In [8]:
circuit = create_bell_circuit()

# 측정할 여러 관측량 정의
obs1 = SparsePauliOp("ZZ")
obs2 = SparsePauliOp("XX")
obs3 = SparsePauliOp("YY")

# V2 PUB 형식: (circuit, [observables list])
estimator = Estimator(mode=local_backend)
job = estimator.run(pubs=[(circuit, [obs1, obs2, obs3])])
result = job.result()

pub_result = result[0]
print("여러 관측량에 대한 기대값:")
print(f"  ZZ: {pub_result.data.evs[0]}")
print(f"  XX: {pub_result.data.evs[1]}")
print(f"  YY: {pub_result.data.evs[2]}")

여러 관측량에 대한 기대값:
  ZZ: 1.0
  XX: 1.0
  YY: -1.0


## 3. 결과 접근하기 (출력 구조 변화)

결과 구조가 크게 바뀌었습니다.

### V1 결과:
```python
# Estimator V1
result.values[0]           # 기대값 배열
result.metadata[0]         # 작업 0의 메타데이터

# Sampler V1  
result.quasi_dists[0]      # 준확률 분포
```

### V2 결과:
```python
# Estimator V2
result[0].data.evs         # PUB 0의 기대값
result[0].data.stds        # PUB 0의 표준편차
result[0].metadata         # PUB 0의 메타데이터

# Sampler V2
result[0].data.meas.get_counts()      # 측정 카운트
result[0].data.meas.get_bitstrings()  # 비트스트링 배열
```

### 예시: V2 Sampler 출력을 V1 스타일 QuasiDistribution으로 변환하기

In [9]:
# V2 Sampler 작업 실행
circuit = create_bell_circuit()
circuit.measure_all()

sampler = Sampler(mode=local_backend)
job = sampler.run([(circuit,)], shots=1024)
result = job.result()

# V2 결과 구조
pub_result = result[0]
shots = pub_result.metadata['shots']
counts = pub_result.data.meas.get_counts()

print(f"V2 카운트 (비트스트링 딕셔너리): {counts}")
print(f"총 shots 수: {shots}")

# V1 스타일 준확률 분포로 변환 (정수 키를 갖는 딕셔너리)
v1_quasi_dist = {int(bitstring, 2): count/shots for bitstring, count in counts.items()}
print(f"\nV1 스타일 QuasiDistribution (마이그레이션용): {v1_quasi_dist}")

V2 카운트 (비트스트링 딕셔너리): {'11': 516, '00': 508}
총 shots 수: 1024

V1 스타일 QuasiDistribution (마이그레이션용): {3: 0.50390625, 0: 0.49609375}


## 4. 옵션 설정하기

V2에서는 옵션 설정이 더 단순해졌습니다.

### V1 방식 (더 이상 권장되지 않음):
```python
from qiskit_ibm_runtime import Options

options = Options()
options.resilience_level = 1
options.execution.shots = 4000
estimator = Estimator(backend=backend, options=options)

# 또는 나중에 갱신:
estimator.set_options(resilience_level=2)
```

### V2 방식 (현재 방식):
primitive 내부에 직접 설정 가능한 `.options` 속성이 내장되어 있습니다.

In [10]:
# 방법 1: 초기화 시 딕셔너리로 options 전달
estimator = Estimator(
    mode=local_backend,
    options={"resilience_level": 1, "default_shots": 2000}
)
print(f"Resilience 수준: {estimator.options.resilience_level}")
print(f"기본 shots 수: {estimator.options.default_shots}")

Resilience 수준: 1
기본 shots 수: 2000


In [11]:
# 방법 2: 속성을 직접 설정 (IDE 자동완성 사용 가능)
estimator = Estimator(mode=local_backend)
estimator.options.resilience_level = 1
estimator.options.default_shots = 4000

print(f"Resilience 수준: {estimator.options.resilience_level}")
print(f"기본 shots 수: {estimator.options.default_shots}")

Resilience 수준: 1
기본 shots 수: 4000


In [12]:
# 방법 3: update()로 여러 값을 한 번에 갱신
estimator = Estimator(mode=local_backend)
estimator.options.update(
    resilience_level=0,
    default_shots=1024
)

print(f"Resilience 수준: {estimator.options.resilience_level}")
print(f"기본 shots 수: {estimator.options.default_shots}")

# 모든 옵션을 딕셔너리로 보기
print(f"\n전체 옵션:\n{asdict(estimator.options)}")

Resilience 수준: 0
기본 shots 수: 1024

전체 옵션:
{'_VERSION': 2, 'max_execution_time': Unset, 'environment': {'log_level': 'WARNING', 'job_tags': None, 'private': False}, 'simulator': {'noise_model': Unset, 'seed_simulator': Unset, 'coupling_map': Unset, 'basis_gates': Unset}, 'default_precision': Unset, 'default_shots': 1024, 'resilience_level': 0, 'seed_estimator': Unset, 'dynamical_decoupling': {'enable': Unset, 'sequence_type': Unset, 'extra_slack_distribution': Unset, 'scheduling_method': Unset, 'skip_reset_qubits': Unset}, 'resilience': {'measure_mitigation': Unset, 'measure_noise_learning': {'num_randomizations': Unset, 'shots_per_randomization': Unset}, 'zne_mitigation': Unset, 'zne': {'amplifier': Unset, 'noise_factors': Unset, 'extrapolator': Unset, 'extrapolated_noise_factors': Unset}, 'pec_mitigation': Unset, 'pec': {'max_overhead': Unset, 'noise_gain': Unset}, 'layer_noise_learning': {'max_layers_to_learn': Unset, 'shots_per_randomization': Unset, 'num_randomizations': Unset, 'la

## 5. 요약: 마이그레이션 체크리스트

### EstimatorV2로 옮기는 단계:

1. **import 갱신:**
   ```python
   # 이전: from qiskit_ibm_runtime import Estimator
   # 이후:
   from qiskit_ibm_runtime import EstimatorV2 as Estimator
   ```

2. **Options import 제거** (더 이상 필요 없음)

3. **backend 인자 변경:**
   ```python
   # 이전: estimator = Estimator(backend=backend)
   # 이후:
   estimator = Estimator(mode=backend)
   ```

4. **옵션 설정 방식 변경:**
   ```python
   # 이전: estimator.set_options(resilience_level=1)
   # 이후:
   estimator.options.resilience_level = 1
   ```

5. **입력을 PUB로 묶기:**
   ```python
   # 이전: job = estimator.run(circuits=[circuit], observables=[obs])
   # 이후:
   job = estimator.run(pubs=[(circuit, obs)])
   ```

6. **결과 접근 방식 변경:**
   ```python
   # 이전: result.values[0]
   # 이후:
   result[0].data.evs
   ```

### SamplerV2로 옮기는 단계:

1. **import 갱신:**
   ```python
   from qiskit_ibm_runtime import SamplerV2 as Sampler
   ```

2. **backend 인자 변경:**
   ```python
   sampler = Sampler(mode=backend)
   ```

3. **옵션 갱신:**
   ```python
   sampler.options.default_shots = 4096
   ```

4. **입력을 PUB로 묶기:**
   ```python
   # 이전: job = sampler.run(circuits=[circuit])
   # 이후:
   job = sampler.run(pubs=[(circuit,)])  # 쉼표에 주의!
   ```

5. **결과 접근 방식 변경:**
   ```python
   # 이전: result.quasi_dists[0]
   # 이후:
   result[0].data.meas.get_counts()
   # 또는
   result[0].data.meas.get_bitstrings()
   ```

## 결론

V2 Primitives의 장점은 다음과 같습니다.
- **더 높은 유연성**: PUB 시스템 활용
- **더 깔끔한 API**: 내장 옵션 제공
- **더 나은 지원**: 다양한 workload 지원 (회로별로 다른 shots, 관측량, 파라미터)
- **향상된 성능** 및 오류 완화 기능

이 노트북의 모든 코드는 클라우드 접근 없이 로컬에서 실행되므로, 마이그레이션을 연습하고 테스트하기 쉽습니다!

---

## 연습 문제

**1. V2 Primitives에서 `run()` 메서드 입력의 필수 구조는 무엇인가요?**

A) 회로와 관측량에 대한 별도의 리스트 (예: `circuits=[...], observables=[...]`)

B) 각 PUB가 튜플인 PUBs(Primitive Unified Blocs)의 단일 리스트

C) 회로를 파라미터에 매핑한 딕셔너리

D) 표준 QuantumCircuit 객체만

***정답:***
<Details>
<br/>
B) 각 PUB가 튜플인 PUBs(Primitive Unified Blocs)의 단일 리스트
</Details>

---

**2. V2 Sampler에서 얻은 결과 객체 `pub_result`가 있을 때, 측정 카운트 딕셔너리에 어떻게 접근하나요?**

A) `pub_result.get_counts()`

B) `pub_result.data.meas.get_counts()`

C) `pub_result.quasi_dists[0]`

D) `pub_result.counts`

***정답:***
<Details>
<br/>
B) `pub_result.data.meas.get_counts()`
</Details>

---

**3. shots와 관련하여 V2 PUB 시스템의 핵심 장점은 무엇인가요?**

A) 같은 작업 안에서 각 회로마다 서로 다른 shot 수를 지정할 수 있다

B) 최적의 shot 수를 자동 계산해 준다

C) 무한한 shots를 허용한다

D) shot 수를 무시하고 항상 백엔드 기본값만 사용한다

***정답:***
<Details>
<br/>
A) 같은 작업 안에서 각 회로마다 서로 다른 shot 수를 지정할 수 있다
</Details>